# 02 Clustering: customer types

k-means on each customer's April to September behaviour. Bijak & Thomas (2012) found that splitting customers
into segments did not considerably improve credit scoring, so the clusters are used mainly to **test the metric**,
not to raise it (`../research/03-clustering.md`):

1. **Metrics within each cluster**: is the hazard model good everywhere, or only on easy customers?
2. **Leave one cluster out**: refit without a customer type, score on it. This is the brief's "leave out some types
   of data and observe your performance".

**Leakage rules:** clusters are learned from the training customers only (validation customers are assigned to the
nearest centre), and the October label is never an input; it is used afterwards, only to describe the clusters.

## Where the code comes from

* k-means and the silhouette: scikit-learn's [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
  and its example [*Selecting the number of clusters with silhouette analysis*](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html).
* Confidence intervals: SciPy's [`scipy.stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)
  with `paired=True`, so each resample keeps a customer's label and prediction together.
* The hazard model is the logit from `01-hazard-models.ipynb`; the leave-one-cluster-out loop is my own.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import bootstrap
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import prep

wide = prep.load_split()
feats = {part: prep.customer_features(df) for part, df in wide.items()}
feats["train"].describe().T[["mean", "std", "min", "max"]].round(2)

## 1. Choose k

Silhouette (Rousseeuw, 1987) on a 5,000-customer sample, and the size of the smallest cluster. A higher silhouette
means better-separated clusters, but the clusters also have to make sense to a person.

In [ ]:
def fit_clusters(train_features, k):
    """Standardise then k-means, both learned from the training customers only."""
    return make_pipeline(StandardScaler(), KMeans(n_clusters=k, n_init=20, random_state=prep.SEED)).fit(train_features)


rows = []
for k in range(2, 9):
    pipe = fit_clusters(feats["train"], k)
    labels = pipe[-1].labels_
    rows.append({"k": k, "silhouette": silhouette_score(pipe[0].transform(feats["train"]), labels,
                                                        sample_size=5000, random_state=prep.SEED),
                 "smallest cluster": int(np.bincount(labels).min())})
pd.DataFrame(rows).round(3)

## 2. Fit the chosen k and describe the clusters

`K = 5` to start with (the research notes expected 4 to 6); change it after looking at section 1. Name each cluster
from the profile, e.g. "pays in full", "revolves", "inactive", "late".

In [ ]:
K = 5
pipe = fit_clusters(feats["train"], K)
cluster = {part: pd.Series(pipe.predict(f), index=f.index, name="cluster") for part, f in feats.items()}

profile = feats["train"].groupby(cluster["train"]).mean()
profile["customers"] = cluster["train"].value_counts().sort_index()
profile["default rate (Oct)"] = wide["train"][prep.Y].groupby(cluster["train"]).mean()   # description only
profile.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(profile.index, profile["default rate (Oct)"], width=0.6, color="#2a78d6")
for c, (rate, n) in profile[["default rate (Oct)", "customers"]].iterrows():
    ax.annotate(f"{rate:.0%}\n{n:,} customers", (c, rate), xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=8, color="#555555")
ax.set_xticks(profile.index, [f"cluster {c}" for c in profile.index])
ax.set_ylabel("October default rate")
ax.set_ylim(0, profile["default rate (Oct)"].max() * 1.3)
ax.set_title("Figure 3: default rate by cluster (training customers)", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
fig.tight_layout()
plt.show()

## 3. The hazard model within each cluster

The logit hazard from `01` (repeated here in two short functions so this notebook runs on its own), fitted once on all
training customers. Within a cluster the "easy separation between
clusters" part of the AUC is removed. The 95% interval is a bootstrap over the cluster's validation customers.

In [ ]:
def fit_logit(data):
    """The logit hazard of 01 (no clustered errors needed: only predictions are used here)."""
    return sm.GLM(data["event"], sm.add_constant(data[prep.features(data)]), family=sm.families.Binomial()).fit()


def predict_october(model, data):
    rows = prep.october(data)
    return pd.Series(np.asarray(model.predict(sm.add_constant(rows[prep.features(rows)], has_constant="add"))),
                     index=rows.index)


def bootstrap_auc(y, p):
    """95% percentile interval for the AUC, resampling customers (label and prediction together)."""
    ci = bootstrap((y.to_numpy(), p.to_numpy()), roc_auc_score, paired=True, vectorized=False,
                   n_resamples=500, method="percentile", random_state=prep.SEED).confidence_interval
    return ci.low, ci.high


pm = prep.person_months(wide)
train, val = pm["train"], pm["validation"]
model = fit_logit(train)
p_val = predict_october(model, val)
y_val = prep.october(val)["event"]
c_val = cluster["validation"].loc[y_val.index]


rows = []
for c in range(K):
    y_c, p_c = y_val[c_val == c], p_val[c_val == c]
    low, high = bootstrap_auc(y_c, p_c)
    rows.append({"cluster": c, "customers": len(y_c), "default rate": y_c.mean(),
                 "AUC": roc_auc_score(y_c, p_c), "95% low": low, "95% high": high})
within = pd.DataFrame(rows).set_index("cluster")
print("overall validation AUC:", round(roc_auc_score(y_val, p_val), 4))
within.round(3)

## 4. Leave one cluster out

For each cluster: refit the hazard model on the training customers **not** in it, then score the validation customers
who **are** in it, a type of customer the model has never seen. A large drop means the model does not generalise to
that type. The same loop works for Oscar's and Louis's models if their fitting code is a function.

In [ ]:
c_train = cluster["train"].loc[train.index]          # each person-month row gets its customer's cluster
rows = []
for c in range(K):
    without_c = fit_logit(train[(c_train != c).to_numpy()])
    y_c = y_val[c_val == c]
    rows.append({"left out": c,
                 "AUC, cluster in training": roc_auc_score(y_c, p_val[c_val == c]),
                 "AUC, cluster left out": roc_auc_score(y_c, predict_october(without_c, val)[c_val == c])})
loco = pd.DataFrame(rows).set_index("left out")
loco["change"] = loco["AUC, cluster left out"] - loco["AUC, cluster in training"]
loco.round(4)

## Notes

* k-means depends on the features chosen, k and the seed. Rerun with another `SEED` and with k - 1 and k + 1, and
  say how much the clusters move.
* A cluster with few defaults gives a wide AUC interval; quote the interval, not just the point.
* Whether clusters **improve** prediction (a cluster covariate, or one model per cluster) is in the earlier draft
  `05-clustering.md`, section 7; it is left out here to keep this simple.